# Stage C3 — Hyperparameter Optimization

Dual-fuel PINN pipeline | Sandrine Schueller Mafra | PPGEM – UFPR
Supports dissertation Sec. 3.5.1 (Table 8), feedback item suggesting
Optuna over a grid search.

**The Table 8 vs. Sec. 3.5.1 gap, resolved by using Optuna:** the prose
in 3.5.1 describes a 3-factor grid (learning rate x batch size x *one*
shared physics-weight scaling factor = 36 combinations), while Table 8
independently lists 13 dimensions, including **four separate** physics-
weight categories. A grid over all of them isn't practical; that's
exactly why the feedback suggested Optuna's TPE sampler instead — this
notebook searches the Table 8 space directly.

**Architecture: fixed from B1 (design decision).** Table 8 also lists
the number of layers, the neurons per layer and the activation. This
notebook does **not** search them: the PINN uses B1's selected
architecture with `tanh` (Sec. 3.2.1.3) and the search covers the other
10 dimensions. Reasons:
- *Clean comparison:* the Phase D comparison with B2 then differs only
  in the physics terms, so any difference can be attributed to them
  (an ablation changes one factor at a time).
- *Small data:* with 34 CV points (~27 per training fold) and fold-to-
  fold spread of the order of the mean error (B1), a larger search
  space raises the risk of fitting the fold noise instead of finding a
  genuinely better configuration.
- *Selection quality:* B1 chose the architecture with 240 fits and a
  paired statistical test; re-choosing it here, with fewer fits per
  configuration and mixed with ten other hyperparameters, would be a
  weaker selection that could contradict B1's.
- *Compatibility:* Table 8's ReLU option has a zero second derivative,
  which makes the HC-λ convexity constraint (Eq. 3.11) trivially
  satisfied.
- *Cost:* each trial is expensive, so a smaller space makes better use
  of the same number of trials.

The possibility that the physics loss favors a different capacity is
addressed by a sensitivity check after C4 (training the PINN on one or
two architectures B1 found statistically equivalent), not by searching
the architecture here. **Table 8 and Sec. 3.5.1 must state that the
architecture is fixed by B1, with this rationale.**

**ReLU caveat (only relevant if `SEARCH_ARCHITECTURE` is switched on):** C1's HC-λ
convexity constraint (Eq. 3.11) needs a **second** derivative, and ReLU
is piecewise linear, so trials with ReLU show a near-zero HC-λ penalty
*by construction*. If the search picks ReLU, check HC-λ convexity
empirically before trusting that term.

**Physics schedule inside each trial:** each trial trains with the same
sigmoid ramp as C4 (Eq. 3.20), compressed to the trial's shorter budget,
and the validation score is taken only once the physics weight is at
full strength (Section 4) — otherwise early stopping could select weights
from before the physics terms took effect.

**Framework note:** the composite loss (C2) depends on collocation-
point *inputs*, not just training pairs, so `model.fit()` can't be used
directly — this notebook writes a small custom training loop
(`tf.GradientTape` around the weights, on top of C2's own tapes around
the inputs). The physics terms are C2's, copied verbatim.

**Input:** `data/masters_data.xlsx`, `outputs/B1_selected_architecture.json`,
`outputs/C1_collocation_points.csv`, `outputs/C1_constraint_config.json`,
`outputs/C2_loss_config.json`
**Output:** every figure and result table is saved to `outputs/html/` as
`C3_<section>[_qualifier].html` (the full trial log is
`C3_06_trials_log.html`); `outputs/C3_best_config.json` (read by C4) and
`outputs/C3_trials_log.csv`.

**Runtime:** `N_TRIALS` x 5 folds x a full physics-informed training
run each — much slower per fit than B1 (the composite loss needs
several nested gradient tapes per step). `N_TRIALS = 20` is a smoke-test
scale; raise it once the notebook runs end-to-end.

## Setup

In [ ]:
import json
import time
import numpy as np
import polars as pl
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path
from sklearn.model_selection import KFold
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import optuna

print("polars    ", pl.__version__)
import plotly
print("plotly    ", plotly.__version__)
print("tensorflow", tf.__version__)
print("optuna    ", optuna.__version__)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "code" else Path.cwd()
RAW_PATH = PROJECT_ROOT / "data" / "masters_data.xlsx"
OUT_DIR = PROJECT_ROOT / "outputs"
SEED = 42
RAW_PATH

## Color palette and output naming (shared across the whole pipeline)

`SPLIT_COLORS` (semantic role: train / validation / test / reference
value / alert / neutral) and `VARIABLE_COLORS` (identity of each of the
4 inputs and 5 outputs) are identical in every A/B/C notebook, so the
same element always has the same color in any chart of the pipeline.

Every figure or result table generated below is also saved to
`outputs/html/`, named `C3_<section>[_qualifier].html` — the number
matches the corresponding section header, so the order in which each
output was produced can be read from the file name alone.

In [ ]:
SPLIT_COLORS = {
    "train": "#B7C9DA",
    "validation": "#2B6EFF",
    "test": "#571D99",
    "reference": "#343A40",   # value transcribed from the dissertation text
    "alert": "#E85D04",       # outlier / out of range / anomaly
    "neutral": "#B0AFA8",     # grid lines / neutral reference
}
VARIABLE_COLORS = {
    "SOI": "#073b3a", "lambda": "#0b6e4f", "sub_rate": "#08a045", "P_rail": "#6bbf59",
    "NOx": "#c7adff", "PM": "#916dd5", "eta": "#7151a9", "HC": "#573d7f", "CO2": "#46325d",
}

HTML_DIR = OUT_DIR / "html"
HTML_DIR.mkdir(parents=True, exist_ok=True)


def flagged_table_html(df, title, out_path, flag_col=None, is_flagged=lambda v: False, ref_cols=()):
    """Result table -> Plotly go.Table -> HTML.
    Columns listed in ref_cols get the 'reference' tone in the header
    (values transcribed from the dissertation text). Cells in flag_col
    get the 'alert' tone wherever is_flagged(value) is True."""
    cols = list(df.columns)
    n = df.shape[0]
    header_fill = [SPLIT_COLORS["reference"] if c in ref_cols else "#F1F3F5" for c in cols]
    header_font = ["white" if c in ref_cols else "black" for c in cols]
    cell_fill = []
    for c in cols:
        if c == flag_col:
            cell_fill.append([SPLIT_COLORS["alert"] if is_flagged(v) else "white"
                               for v in df[c].to_list()])
        else:
            cell_fill.append(["white"] * n)
    fig = go.Figure(data=[go.Table(
        header=dict(values=cols, fill_color=header_fill,
                     font=dict(color=header_font), align="left"),
        cells=dict(values=[df[c].to_list() for c in cols],
                    fill_color=cell_fill, align="left"),
    )])
    fig.update_layout(title=title, margin=dict(t=40, l=10, r=10, b=10))
    fig.write_html(str(out_path), include_plotlyjs="inline")
    return fig


def simple_table_html(df, title, out_path):
    return flagged_table_html(df, title, out_path)

# Constraint identity (not covered by the two palettes above): each constraint takes the
# color of the output it constrains, and the dash pattern tells same-output constraints apart.
CONSTRAINT_STYLE = {
    "NOx-SOI": dict(color=VARIABLE_COLORS["NOx"], dash="solid"),
    "PM-lambda": dict(color=VARIABLE_COLORS["PM"], dash="solid"),
    "HC-lambda": dict(color=VARIABLE_COLORS["HC"], dash="solid"),
    "NOx-PM": dict(color=VARIABLE_COLORS["PM"], dash="dash"),
    "eta-NOx": dict(color=VARIABLE_COLORS["eta"], dash="solid"),
    "non-negativity": dict(color=SPLIT_COLORS["reference"], dash="dot"),
}

## 1. Load data, B1's architecture, C1's collocation points and C2's calibrated weights

The loading code is C2's, copied verbatim (same split, same
normalization, same checks against C1's configuration), followed by the
CV pool (train + val, test held out) and C2's loss configuration.

In [ ]:
COLUMN_MAP = {
    "SOI [o.CA]": "SOI", "Lambda [-]": "lambda", "Sub. Rate [%]": "sub_rate",
    "Prail [bar]": "P_rail", "HC [g/kW.h]": "HC", "NOX [ppm]": "NOx",
    "CO2 [%]": "CO2", "SO_H [FSN]": "PM", "ETA [%]": "eta",
}
INPUT_COLS = ["SOI", "lambda", "sub_rate", "P_rail"]
OUTPUT_COLS = ["HC", "NOx", "CO2", "PM", "eta"]
ALL_COLS = INPUT_COLS + OUTPUT_COLS
N_IN, N_OUT = len(INPUT_COLS), len(OUTPUT_COLS)
SOI_IDX, LAMBDA_IDX, SUBRATE_IDX, PRAIL_IDX = [INPUT_COLS.index(c) for c in INPUT_COLS]
HC_IDX, NOX_IDX, CO2_IDX, PM_IDX, ETA_IDX = [OUTPUT_COLS.index(c) for c in OUTPUT_COLS]
EMISSION_IDXS = [HC_IDX, NOX_IDX, CO2_IDX, PM_IDX]

df = pl.read_excel(RAW_PATH).rename(COLUMN_MAP).select(ALL_COLS)
n = df.shape[0]
medians = {c: df[c].median() for c in INPUT_COLS}
ranges = {c: (df[c].max() - df[c].min()) for c in INPUT_COLS}
deviation = np.column_stack([np.abs(df[c].to_numpy() - medians[c]) / ranges[c] for c in INPUT_COLS])
raw_block = np.array(INPUT_COLS)[deviation.argmax(axis=1)]

def smooth_isolated_labels(labels, passes=2):
    out = list(labels)
    for _ in range(passes):
        changed = False
        for i in range(1, len(out) - 1):
            if out[i] != out[i - 1] and out[i - 1] == out[i + 1]:
                out[i] = out[i - 1]
                changed = True
        if not changed:
            break
    return np.array(out)

ofat_block = smooth_isolated_labels(raw_block)
extremity = np.zeros(n)
for b in np.unique(ofat_block):
    idx = np.where(ofat_block == b)[0]
    vals = df[b].to_numpy()[idx]
    order = np.argsort(vals)
    m = len(idx)
    pos = np.array([0.5]) if m == 1 else np.empty(m)
    if m > 1:
        ranks = np.empty(m)
        ranks[order] = np.arange(m)
        pos = ranks / (m - 1)
    extremity[idx] = np.abs(pos - 0.5) * 2
rng_np = np.random.default_rng(SEED)
jitter = rng_np.uniform(-1e-9, 1e-9, size=n)
order = np.argsort(-(extremity + jitter))
split = np.array(["train"] * n)
split[order[:6]] = "test"
split[order[6:12]] = "val"
df = df.with_columns(pl.Series("split", split))
train_df = df.filter(pl.col("split") == "train")
train_min = {c: train_df[c].min() for c in ALL_COLS}
train_max = {c: train_df[c].max() for c in ALL_COLS}
df = df.with_columns([
    ((pl.col(c) - train_min[c]) / (train_max[c] - train_min[c])).alias(f"{c}_norm")
    for c in ALL_COLS
])
X_train = df.filter(pl.col("split") == "train").select([f"{c}_norm" for c in INPUT_COLS]).to_numpy().astype(np.float32)
Y_train = df.filter(pl.col("split") == "train").select([f"{c}_norm" for c in OUTPUT_COLS]).to_numpy().astype(np.float32)

# ---- C1 outputs: collocation points + constraint configuration ----
colloc_path = OUT_DIR / "C1_collocation_points.csv"
config_path = OUT_DIR / "C1_constraint_config.json"
for p in (colloc_path, config_path):
    if not p.exists():
        raise FileNotFoundError(f"{p} not found -- run C1 (including its save cell) before C2.")
colloc_df = pl.read_csv(colloc_path)
X_colloc = colloc_df.select(INPUT_COLS).to_numpy().astype(np.float32)
rho_colloc = colloc_df["density_ratio"].to_numpy()
validity_eff_nox = colloc_df["validity_eff_nox"].to_numpy()
with open(config_path) as f:
    C1_CONFIG = json.load(f)

# the non-negativity floors must be the ones C1 computed from this same split
EMISSIONS = [OUTPUT_COLS[i] for i in EMISSION_IDXS]
floors_here = [float(-train_min[c] / (train_max[c] - train_min[c])) for c in EMISSIONS]
floors_c1 = [C1_CONFIG["nonneg_floor_norm"][c] for c in EMISSIONS]
if not np.allclose(floors_here, floors_c1):
    raise ValueError("non-negativity floors differ from C1 -- the data or split changed; rerun C1.")

# ---- B1 architecture ----
selection_path = OUT_DIR / "B1_selected_architecture.json"
if not selection_path.exists():
    raise FileNotFoundError(f"{selection_path} not found -- run B1 (including its save cell) before C2.")
with open(selection_path) as f:
    selection = json.load(f)
HIDDEN_UNITS = tuple(selection["hidden_units"])
print(f"Loaded {X_colloc.shape[0]} collocation points from C1; architecture {selection['architecture_id']} {HIDDEN_UNITS}")
print(f"C1 config: NOx-SOI direction = {C1_CONFIG['nox_soi_direction']}, validity rule = {C1_CONFIG['validity_eff_nox_rule']}")

# ---- CV pool (train + val); test stays held out ----
cv_pool = df.filter(pl.col("split") != "test")
X_cv = cv_pool.select([f"{c}_norm" for c in INPUT_COLS]).to_numpy().astype(np.float32)
Y_cv = cv_pool.select([f"{c}_norm" for c in OUTPUT_COLS]).to_numpy().astype(np.float32)
print("CV pool:", X_cv.shape[0], "points (test held out)")

# ---- C2's calibrated weights and schedule ----
c2_path = OUT_DIR / "C2_loss_config.json"
if not c2_path.exists():
    raise FileNotFoundError(f"{c2_path} not found -- run C2 (including its save cell) before C3.")
with open(c2_path) as f:
    loss_config = json.load(f)
if loss_config["c1_config"] != C1_CONFIG:
    raise ValueError("C2 was calibrated with a different C1 configuration -- rerun C2.")
w_final_base = loss_config["w_final"]
K_SCHEDULE, T0_SCHEDULE = loss_config["k_schedule"], loss_config["t0_schedule"]
print(f"C2 calibration rule: {loss_config['calibration_mode']}; schedule k={K_SCHEDULE}, t0={T0_SCHEDULE}")

## 2. Table 8 search space

The four physics-weight hyperparameters are **scaling factors on C2's
calibrated $w_j^{\text{final}}$**, one shared per constraint *category*
(Sec. 3.5.1's prose uses this "scaling factor" language for a single
shared factor; Table 8 splits it into four — implementation choice made
explicit here, not fully pinned down by the text). The table below lists
all 13 Table 8 dimensions: the three architectural ones are fixed to
B1's selection (see the header), the other 10 are searched.

In [ ]:
# Design decision: architecture fixed from B1 (see the header). True would restore the full
# 13-dimension Table 8 search -- not used in the dissertation; B2 would then need retraining.
SEARCH_ARCHITECTURE = False

CONSTRAINT_CATEGORY = {
    "NOx-SOI": "monotonic", "PM-lambda": "monotonic",
    "HC-lambda": "shape",
    "NOx-PM": "tradeoff", "eta-NOx": "tradeoff",
    "non-negativity": "nonneg",
}
CATEGORY_KEY = {"monotonic": "scale_monotonic", "shape": "scale_shape",
                "tradeoff": "scale_tradeoff", "nonneg": "scale_nonneg"}

SEARCH_SPACE = [  # (name, kind, low/choices, high, log, group)
    ("n_layers", "int", 1, 4, False, "architecture"),
    ("n_units", "categorical", [8, 16, 32, 64, 128], None, False, "architecture"),
    ("activation", "categorical", ["tanh", "sigmoid", "relu"], None, False, "architecture"),
    ("learning_rate", "float", 1e-5, 1e-2, True, "optimization"),
    ("batch_size", "categorical", [8, 16, 32], None, False, "optimization"),
    ("beta_1", "float", 0.85, 0.95, False, "optimization"),
    ("beta_2", "float", 0.99, 0.999, False, "optimization"),
    ("scale_monotonic", "float", 0.01, 10.0, True, "physics weights"),
    ("scale_shape", "float", 0.01, 10.0, True, "physics weights"),
    ("scale_tradeoff", "float", 0.001, 5.0, True, "physics weights"),
    ("scale_nonneg", "float", 0.1, 20.0, True, "physics weights"),
    ("weight_decay", "float", 1e-6, 1e-3, True, "regularization"),
    ("dropout_rate", "float", 0.0, 0.5, False, "regularization"),
]


def suggest_hyperparameters(trial):
    hp = {}
    for name, kind, lo, hi, log, group in SEARCH_SPACE:
        if group == "architecture" and not SEARCH_ARCHITECTURE:
            continue
        if kind == "int":
            hp[name] = trial.suggest_int(name, lo, hi)
        elif kind == "categorical":
            hp[name] = trial.suggest_categorical(name, lo)
        else:
            hp[name] = trial.suggest_float(name, lo, hi, log=log)
    if SEARCH_ARCHITECTURE:
        hp["hidden_units"] = [hp["n_units"]] * hp["n_layers"]
    else:
        hp["hidden_units"], hp["activation"] = list(HIDDEN_UNITS), "tanh"
    return hp


space_table = pl.DataFrame([{
    "hyperparameter": name, "group": group,
    "range_or_choices": str(lo) if kind == "categorical" else f"{lo} - {hi}",
    "log_scale": log,
    "searched": bool(group != "architecture" or SEARCH_ARCHITECTURE),
    "fixed_value": "" if (group != "architecture" or SEARCH_ARCHITECTURE)
                   else {"n_layers": str(len(HIDDEN_UNITS)), "n_units": str(list(HIDDEN_UNITS)),
                         "activation": "tanh"}[name],
} for name, kind, lo, hi, log, group in SEARCH_SPACE])
simple_table_html(space_table, f"C3 -- Table 8 search space (architecture searched: {SEARCH_ARCHITECTURE})",
                   HTML_DIR / "C3_02_search_space.html")
space_table

## 3. Model builder (with dropout) and the six physics penalties

The model is B1's (copied verbatim, η sigmoid + `Rescaling` head); the
search version only adds a `Dropout` layer after each hidden layer and
lets the activation vary when the architecture is searched. The check
below confirms that, with dropout 0 and `tanh`, it has exactly the
parameters B1 recorded. The physics penalties are C2's, copied verbatim
(C1's functions + C2's per-point versions, `physics_loss`, and the
regularization term).

In [ ]:
assert OUTPUT_COLS[-1] == "eta", "the eta head is appended last; OUTPUT_COLS must end with eta"

# eta as a physical fraction (A1 Section 7: stored as a fraction) and its train-only range (A3 Section 9)
to_fraction = (lambda v: v / 100) if df["eta"].max() > 1 else (lambda v: v)
ETA_MIN, ETA_MAX = to_fraction(train_min["eta"]), to_fraction(train_max["eta"])
ETA_MEAN_TRAIN = float(np.mean(to_fraction(df.filter(pl.col("split") == "train")["eta"].to_numpy())))
ETA_BIAS_INIT = float(np.log(ETA_MEAN_TRAIN / (1 - ETA_MEAN_TRAIN)))    # logit(mean train eta)


def build_model(hidden_units, seed, input_dim=N_IN, output_dim=N_OUT):
    tf.random.set_seed(seed)
    inputs = keras.Input(shape=(input_dim,))
    x = inputs
    for units in hidden_units:
        x = layers.Dense(units, activation="tanh")(x)
    emissions = layers.Dense(output_dim - 1, activation="linear", name="emissions")(x)   # HC, NOx, CO2, PM
    eta_frac = layers.Dense(1, activation="sigmoid", name="eta_fraction",
                            bias_initializer=keras.initializers.Constant(ETA_BIAS_INIT))(x)
    eta_norm = layers.Rescaling(scale=1.0 / (ETA_MAX - ETA_MIN),
                                offset=-ETA_MIN / (ETA_MAX - ETA_MIN), name="eta_rescaled")(eta_frac)
    outputs = layers.Concatenate(name="outputs")([emissions, eta_norm])
    model = keras.Model(inputs=inputs, outputs=outputs)
    model.compile(optimizer="adam", loss="mse")
    return model


def predict_np(model, X):
    """Forward pass as a NumPy array, without model.predict().

    model.predict() builds a new tf.function for every freshly built model;
    in a loop of 240 fits that triggers TensorFlow's "tf.function retracing"
    warning and is slower than a direct call on arrays this small. A direct
    call with training=False gives the same predictions (no dropout or batch
    normalization in these models)."""
    return np.asarray(model(np.asarray(X, dtype="float32"), training=False))


def build_model_hp(hidden_units, activation, dropout_rate, seed, input_dim=N_IN, output_dim=N_OUT):
    tf.random.set_seed(seed)
    inputs = keras.Input(shape=(input_dim,))
    x = inputs
    for units in hidden_units:
        x = layers.Dense(units, activation=activation)(x)
        if dropout_rate > 0:
            x = layers.Dropout(dropout_rate)(x)
    emissions = layers.Dense(output_dim - 1, activation="linear", name="emissions")(x)
    eta_frac = layers.Dense(1, activation="sigmoid", name="eta_fraction",
                            bias_initializer=keras.initializers.Constant(ETA_BIAS_INIT))(x)
    eta_norm = layers.Rescaling(scale=1.0 / (ETA_MAX - ETA_MIN),
                                offset=-ETA_MIN / (ETA_MAX - ETA_MIN), name="eta_rescaled")(eta_frac)
    outputs = layers.Concatenate(name="outputs")([emissions, eta_norm])
    return keras.Model(inputs=inputs, outputs=outputs)


m_b1 = build_model(HIDDEN_UNITS, seed=SEED)
m_c3 = build_model_hp(HIDDEN_UNITS, "tanh", 0.0, seed=SEED)
model_check = pl.DataFrame({
    "check": ["parameters, B1 builder", "parameters, C3 builder (dropout 0, tanh)",
              "parameters recorded by B1", "C3 builder == B1"],
    "value": [str(m_b1.count_params()), str(m_c3.count_params()), str(selection["n_params"]),
              str(m_b1.count_params() == m_c3.count_params() == selection["n_params"])],
})
flagged_table_html(model_check, f"C3 -- Search model vs. B1 model ({selection['architecture_id']})",
                    HTML_DIR / "C3_03_model_check.html", flag_col="value", is_flagged=lambda v: v == "False")
assert m_b1.count_params() == m_c3.count_params() == selection["n_params"]
model_check

In [ ]:
# ---- C1's constraint settings (from C1_constraint_config.json) ----
NOX_SOI_DIRECTION = C1_CONFIG["nox_soi_direction"]
PM_LAMBDA_DIRECTION = C1_CONFIG["pm_lambda_direction"]
NONNEG_FLOOR_NORM = floors_c1

# ---- C1's constraint functions, copied verbatim ----
def monotonic_constraint(predict_fn, x, out_idx, in_idx, direction):
    x_t = tf.convert_to_tensor(x, dtype=tf.float32)
    with tf.GradientTape() as tape:
        tape.watch(x_t)
        y = predict_fn(x_t)
        target = y[:, out_idx]
    grad = tape.gradient(target, x_t)
    d = grad[:, in_idx]
    violation = d if direction == "decreasing" else -d    # the derivative sign that is NOT allowed
    return tf.reduce_mean(tf.square(tf.maximum(0.0, violation)))


def nox_soi_constraint(predict_fn, x, direction=None):
    return monotonic_constraint(predict_fn, x, NOX_IDX, SOI_IDX, direction or NOX_SOI_DIRECTION)


def pm_lambda_constraint(predict_fn, x):
    return monotonic_constraint(predict_fn, x, PM_IDX, LAMBDA_IDX, PM_LAMBDA_DIRECTION)


def convexity_constraint(predict_fn, x, out_idx, in_idx):
    x_t = tf.convert_to_tensor(x, dtype=tf.float32)
    with tf.GradientTape() as tape2:
        tape2.watch(x_t)
        with tf.GradientTape() as tape1:
            tape1.watch(x_t)
            y = predict_fn(x_t)
            target = y[:, out_idx]
        grad1 = tape1.gradient(target, x_t)
        d_first = grad1[:, in_idx]
    grad2 = tape2.gradient(d_first, x_t)
    d_second = grad2[:, in_idx]
    return tf.reduce_mean(tf.square(tf.maximum(0.0, -d_second)))


def hc_lambda_constraint(predict_fn, x):
    return convexity_constraint(predict_fn, x, HC_IDX, LAMBDA_IDX)


def tradeoff_constraint(predict_fn, x, out_idx_a, out_idx_b, in_idx):
    x_t = tf.convert_to_tensor(x, dtype=tf.float32)
    with tf.GradientTape(persistent=True) as tape:
        tape.watch(x_t)
        y = predict_fn(x_t)
        a = y[:, out_idx_a]
        b = y[:, out_idx_b]
    grad_a = tape.gradient(a, x_t)[:, in_idx]
    grad_b = tape.gradient(b, x_t)[:, in_idx]
    del tape
    return tf.reduce_mean(tf.square(tf.maximum(0.0, grad_a * grad_b)))


def nox_pm_tradeoff_constraint(predict_fn, x):
    return tradeoff_constraint(predict_fn, x, NOX_IDX, PM_IDX, SOI_IDX)


def eff_nox_tradeoff_constraint(predict_fn, x):
    return tradeoff_constraint(predict_fn, x, ETA_IDX, NOX_IDX, SOI_IDX)


def nonneg_constraint(predict_fn, x, out_idxs=EMISSION_IDXS, floors=NONNEG_FLOOR_NORM):
    x_t = tf.convert_to_tensor(x, dtype=tf.float32)
    y = predict_fn(x_t)
    emissions = tf.gather(y, out_idxs, axis=1)
    floor_t = tf.constant(floors, dtype=tf.float32)          # normalized value of physical zero
    # Eq. 3.14: sum over the four emissions, mean over points
    return tf.reduce_mean(tf.reduce_sum(tf.square(tf.maximum(0.0, floor_t - emissions)), axis=1))

In [ ]:
def g_monotonic(predict_fn, x, out_idx, in_idx, direction):
    x_t = tf.convert_to_tensor(x, dtype=tf.float32)
    with tf.GradientTape() as tape:
        tape.watch(x_t)
        target = predict_fn(x_t)[:, out_idx]
    d = tape.gradient(target, x_t)[:, in_idx]
    return d if direction == "decreasing" else -d

def g_convexity(predict_fn, x, out_idx, in_idx):
    x_t = tf.convert_to_tensor(x, dtype=tf.float32)
    with tf.GradientTape() as tape2:
        tape2.watch(x_t)
        with tf.GradientTape() as tape1:
            tape1.watch(x_t)
            target = predict_fn(x_t)[:, out_idx]
        d_first = tape1.gradient(target, x_t)[:, in_idx]
    d_second = tape2.gradient(d_first, x_t)[:, in_idx]
    return -d_second

def g_tradeoff(predict_fn, x, out_idx_a, out_idx_b, in_idx):
    x_t = tf.convert_to_tensor(x, dtype=tf.float32)
    with tf.GradientTape(persistent=True) as tape:
        tape.watch(x_t)
        y = predict_fn(x_t)
        a, b = y[:, out_idx_a], y[:, out_idx_b]
    grad_a = tape.gradient(a, x_t)[:, in_idx]
    grad_b = tape.gradient(b, x_t)[:, in_idx]
    del tape
    return grad_a * grad_b

def g_nonneg(predict_fn, x, out_idxs=EMISSION_IDXS, floors=NONNEG_FLOOR_NORM):
    x_t = tf.convert_to_tensor(x, dtype=tf.float32)
    emissions = tf.gather(predict_fn(x_t), out_idxs, axis=1)
    return tf.constant(floors, dtype=tf.float32) - emissions        # [N, 4]

G_FUNCTIONS = {
    "NOx-SOI": lambda f, x: g_monotonic(f, x, NOX_IDX, SOI_IDX, NOX_SOI_DIRECTION),
    "PM-lambda": lambda f, x: g_monotonic(f, x, PM_IDX, LAMBDA_IDX, PM_LAMBDA_DIRECTION),
    "HC-lambda": lambda f, x: g_convexity(f, x, HC_IDX, LAMBDA_IDX),
    "NOx-PM": lambda f, x: g_tradeoff(f, x, NOX_IDX, PM_IDX, SOI_IDX),
    "eta-NOx": lambda f, x: g_tradeoff(f, x, ETA_IDX, NOX_IDX, SOI_IDX),
    "non-negativity": lambda f, x: g_nonneg(f, x),
}
CONSTRAINT_NAMES = list(G_FUNCTIONS)
GRADIENT_CONSTRAINTS = CONSTRAINT_NAMES[:5]          # these carry lambda_j(x); non-negativity does not

def per_point_penalty(g):
    p = tf.square(tf.maximum(0.0, g))
    return tf.reduce_sum(p, axis=1) if len(g.shape) > 1 else p

def per_point_magnitude(g):
    m = tf.square(g)
    return tf.reduce_sum(m, axis=1) if len(g.shape) > 1 else m

VALIDITY = {name: np.ones(len(X_colloc)) for name in GRADIENT_CONSTRAINTS}
VALIDITY["eta-NOx"] = validity_eff_nox
LAMBDA_X = {name: rho_colloc * VALIDITY[name] for name in GRADIENT_CONSTRAINTS}   # lambda_j^0 = 1 here

def physics_loss(name, predict_fn, x=X_colloc):
    p = per_point_penalty(G_FUNCTIONS[name](predict_fn, x))
    if name in LAMBDA_X:
        return tf.reduce_mean(tf.constant(LAMBDA_X[name], dtype=tf.float32) * p)
    return tf.reduce_mean(p)

In [ ]:
def regularization_loss(model):
    P = model.count_params()
    sq_sum = tf.add_n([tf.reduce_sum(tf.square(w)) for w in model.trainable_weights if len(w.shape) > 1])
    return sq_sum / P


## 4. Custom training step and the physics schedule inside a trial

`model.fit()` can't take collocation points as a second, differently-
shaped input alongside `(X, Y)`, so training is one explicit step: an
outer tape differentiates the (already tape-using) composite loss with
respect to the model's *weights*, then the optimizer applies those
gradients. Weight decay (Eq. 3.19's regularization weight, a Table 8
hyperparameter) is added directly in the loss.

**Schedule inside a trial.** C2's schedule (Eq. 3.20, $k = 0.01$,
$t_0 = 750$) reaches 99 % of full strength only at epoch
$t_{99} = t_0 + \ln(99)/k \approx 1210$, far beyond a trial's budget.
Using it unchanged would keep the physics weight below 0.3 % for the
whole trial — the search would tune the physics scales without physics.
So each trial uses the **same curve, compressed** in time:

$$
t_{\text{equiv}} = \text{epoch} \cdot \frac{t_{99}}{f_{\text{ramp}}\,E_{\text{trial}}},
\qquad w_j(\text{epoch}) = \frac{w_j^{\text{final}}\, s_{c(j)}}{1 + \exp[-k(t_{\text{equiv}} - t_0)]}
$$

with $f_{\text{ramp}} = 0.5$: the physics weight starts at the same
≈ 0.05 % as in C4 and reaches 99 % halfway through the trial.
$s_{c(j)}$ is the trial's scaling factor for constraint $j$'s category.

**Model selection only at full strength.** Validation MSE is tracked,
and early stopping counts patience, only from the epoch where the ramp
reaches 99 %. Without this, a trial whose validation MSE is lowest early
(before the physics terms act) would be scored — and its weights kept —
as if it were a physics-informed model.

In [ ]:
T99_EQUIV = T0_SCHEDULE + np.log(99) / K_SCHEDULE      # epoch at which C2's schedule reaches 99 %
RAMP_FULL_FRACTION = 0.5                                # share of a trial's budget used by the ramp

def scheduled_weight(t, w_j_final, k=K_SCHEDULE, t0=T0_SCHEDULE):
    return w_j_final / (1 + np.exp(-k * (t - t0)))


def trial_schedule_factor(epoch, epochs_budget, ramp_full_fraction=RAMP_FULL_FRACTION):
    t_equiv = epoch * T99_EQUIV / (ramp_full_fraction * epochs_budget)
    return 1.0 / (1 + np.exp(-K_SCHEDULE * (t_equiv - T0_SCHEDULE)))


def composite_loss(model, X, Y, w_now, weight_decay, sigma2):
    predict_fn = lambda x: model(x, training=True)
    X_t = tf.convert_to_tensor(X, dtype=tf.float32)
    Y_t = tf.convert_to_tensor(Y, dtype=tf.float32)
    pred = predict_fn(X_t)
    l_data = tf.reduce_mean(tf.reduce_sum(tf.square(pred - Y_t) / sigma2, axis=1))
    l_phys = 0.0
    for name in CONSTRAINT_NAMES:
        l_phys = l_phys + float(w_now[name]) * physics_loss(name, predict_fn)
    l_reg = weight_decay * regularization_loss(model)
    return l_data + l_phys + l_reg, l_data


def train_step(model, optimizer, X, Y, w_now, weight_decay, sigma2):
    with tf.GradientTape() as tape:
        loss, l_data = composite_loss(model, X, Y, w_now, weight_decay, sigma2)
    grads = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(grads, model.trainable_variables))
    return loss, l_data

In [ ]:
_E = 150  # illustration with the default trial budget below
_ep = np.arange(_E + 1)
_f = np.array([trial_schedule_factor(e, _E) for e in _ep])
_ramp_end = int(np.argmax(_f >= 0.99))
fig = go.Figure()
fig.add_trace(go.Scatter(x=_ep, y=_f, mode="lines", name="physics weight / final (trial)",
                          line=dict(color=SPLIT_COLORS["reference"], width=2.5)))
fig.add_vrect(x0=_ramp_end, x1=_E, fillcolor=SPLIT_COLORS["validation"], opacity=0.12, line_width=0,
              annotation_text="validation tracked / early stopping active", annotation_position="top left")
fig.update_layout(title=f"Physics schedule inside a {_E}-epoch trial (C2's curve, compressed; "
                        f"99 % reached at epoch {_ramp_end})",
                  xaxis_title="trial epoch", yaxis=dict(title="w_j(epoch) / (w_j_final * s_c)", range=[0, 1.05]),
                  width=850, height=400)
fig.show()
fig.write_html(str(HTML_DIR / "C3_04_trial_schedule.html"), include_plotlyjs="inline")

## 5. Objective function — 5-fold CV, mean validation MSE

Per Sec. 3.5.1: the optimization target is plain validation MSE (data
fit, normalized outputs), not the physics-inclusive loss — physics
shapes *training* through Section 4's composite loss, it just isn't what
Optuna scores. Every trial uses the same 5 folds (fixed seed), so trials
are compared on identical splits.

**Logged per trial** (Section 6's trial log): the mean and standard
deviation of the fold MSEs, every fold's MSE, the mean validation MSE
per output, the mean physics penalty per constraint at the selected
weights (unscaled — how well each constraint is satisfied, independent
of its weight), the epochs run per fold and the trial's duration.

**Reading the result:** the folds are drawn from the train+val pool, so
they mostly test *interpolation*, where physics priors rarely lower the
data error. An objective based only on data MSE therefore tends to favor
small physics scales. The logged penalties make that trade-off visible;
a multi-objective study (data MSE and constraint violation) would be the
stricter alternative.

In [ ]:
N_FOLDS = 5
EPOCHS_PER_TRIAL = 150
PATIENCE_PER_TRIAL = 20


def objective(trial):
    t_start = time.time()
    hp = suggest_hyperparameters(trial)
    w_scaled = {name: w_final_base[name] * hp[CATEGORY_KEY[CONSTRAINT_CATEGORY[name]]]
                for name in CONSTRAINT_NAMES}
    fold_mses, fold_out_mse, fold_pen, fold_epochs = [], [], [], []
    kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    for fold_i, (tr_idx, va_idx) in enumerate(kf.split(X_cv)):
        keras.backend.clear_session()
        Xtr, Ytr = X_cv[tr_idx], Y_cv[tr_idx]
        Xva, Yva = X_cv[va_idx], Y_cv[va_idx]
        sigma2 = tf.constant(np.maximum(Ytr.var(axis=0, ddof=1), 1e-8), dtype=tf.float32)
        model = build_model_hp(hp["hidden_units"], hp["activation"], hp["dropout_rate"],
                               seed=SEED * 100 + fold_i)
        optimizer = keras.optimizers.Adam(learning_rate=hp["learning_rate"],
                                          beta_1=hp["beta_1"], beta_2=hp["beta_2"])
        rng = np.random.default_rng(SEED * 1000 + trial.number * 10 + fold_i)
        best_val, patience_ctr, best_weights, epochs_run = np.inf, 0, None, 0
        for epoch in range(EPOCHS_PER_TRIAL):
            factor = trial_schedule_factor(epoch, EPOCHS_PER_TRIAL)
            w_now = {name: w_scaled[name] * factor for name in CONSTRAINT_NAMES}
            perm = rng.permutation(len(Xtr))
            # iterate by start offset so a fold size that doesn't divide evenly by
            # batch_size still trains on every point
            for start in range(0, len(perm), hp["batch_size"]):
                idx = perm[start:start + hp["batch_size"]]
                train_step(model, optimizer, Xtr[idx], Ytr[idx], w_now, hp["weight_decay"], sigma2)
            epochs_run = epoch + 1
            if factor < 0.99:
                continue                      # physics not at full strength yet: no model selection
            val_mse = float(np.mean((predict_np(model, Xva) - Yva) ** 2))
            if val_mse < best_val - 1e-6:
                best_val, patience_ctr, best_weights = val_mse, 0, model.get_weights()
            else:
                patience_ctr += 1
                if patience_ctr >= PATIENCE_PER_TRIAL:
                    break
        if best_weights is not None:
            model.set_weights(best_weights)
        pred_va = predict_np(model, Xva)
        fold_mses.append(float(np.mean((pred_va - Yva) ** 2)))
        fold_out_mse.append(np.mean((pred_va - Yva) ** 2, axis=0))
        eval_fn = lambda x: model(x, training=False)
        fold_pen.append({name: float(physics_loss(name, eval_fn)) for name in CONSTRAINT_NAMES})
        fold_epochs.append(epochs_run)

    trial.set_user_attr("fold_mses", fold_mses)
    trial.set_user_attr("std_mse", float(np.std(fold_mses, ddof=1)))
    trial.set_user_attr("mse_per_output", dict(zip(OUTPUT_COLS, np.mean(fold_out_mse, axis=0).tolist())))
    trial.set_user_attr("physics_penalty", {n: float(np.mean([p[n] for p in fold_pen])) for n in CONSTRAINT_NAMES})
    trial.set_user_attr("epochs_run", fold_epochs)
    trial.set_user_attr("hidden_units", hp["hidden_units"])
    trial.set_user_attr("activation", hp["activation"])
    trial.set_user_attr("duration_s", round(time.time() - t_start, 1))
    return float(np.mean(fold_mses))

## 6. Run the study

In [ ]:
N_TRIALS = 20  # smoke-test scale; raise once the notebook runs cleanly end-to-end

sampler = optuna.samplers.TPESampler(seed=SEED)
study = optuna.create_study(direction="minimize", sampler=sampler, study_name="C3_pinn_hparams")
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

best = study.best_trial
print(f"Best trial: #{best.number}  value={study.best_value:.5f}")

# ---- full trial log (one row per trial) ----
log_rows = []
for t in study.trials:
    if t.value is None:
        continue
    row = {"trial": t.number, "cv_mean_mse": round(t.value, 6), "cv_std_mse": round(t.user_attrs["std_mse"], 6),
           "is_best": t.number == best.number, "hidden_units": str(t.user_attrs["hidden_units"]),
           "activation": t.user_attrs["activation"]}
    row.update({k: (round(v, 6) if isinstance(v, float) else v) for k, v in t.params.items()})
    row.update({f"mse_{o}": round(v, 6) for o, v in t.user_attrs["mse_per_output"].items()})
    row.update({f"penalty_{n}": float(f"{v:.4g}") for n, v in t.user_attrs["physics_penalty"].items()})
    row["epochs_run_per_fold"] = str(t.user_attrs["epochs_run"])
    row["fold_mses"] = str([round(v, 5) for v in t.user_attrs["fold_mses"]])
    row["duration_s"] = t.user_attrs["duration_s"]
    log_rows.append(row)
trials_log = pl.DataFrame(log_rows).sort("cv_mean_mse")
trials_log = pl.DataFrame({"rank": list(range(1, trials_log.shape[0] + 1)),
                           **{c: trials_log[c].to_list() for c in trials_log.columns}})
OUT_DIR.mkdir(parents=True, exist_ok=True)
trials_log.write_csv(OUT_DIR / "C3_trials_log.csv")
simple_table_html(trials_log, f"C3 -- Trial log sorted by CV mean MSE (best = trial {best.number})",
                   HTML_DIR / "C3_06_trials_log.html")
trials_log

## 7. Optimization history

In [ ]:
done = [t for t in study.trials if t.value is not None]
nums = [t.number for t in done]
vals = [t.value for t in done]
best_so_far = np.minimum.accumulate(vals)
fig = go.Figure()
fig.add_trace(go.Scatter(x=nums, y=vals, mode="markers", name="trial CV mean MSE",
                         marker=dict(color=SPLIT_COLORS["validation"], size=9,
                                     line=dict(color=SPLIT_COLORS["reference"], width=0.5))))
fig.add_trace(go.Scatter(x=nums, y=best_so_far, mode="lines", name="best so far",
                         line=dict(color=SPLIT_COLORS["reference"], width=2)))
fig.update_layout(title="Optimization history", xaxis_title="trial",
                  yaxis=dict(title="CV mean validation MSE (log scale)", type="log"), width=800, height=420)
fig.show()
fig.write_html(str(HTML_DIR / "C3_07_optimization_history.html"), include_plotlyjs="inline")

## 8. Parallel coordinates — hyperparameters vs. objective

In [ ]:
numeric_params = [p for p in best.params if isinstance(best.params[p], (int, float))]
dims = []
for p in numeric_params:
    v = [t.params[p] for t in done]
    log_axis = any(p == name and log for name, _, _, _, log, _ in SEARCH_SPACE)
    dims.append(dict(label=f"log10 {p}" if log_axis else p,
                     values=np.log10(v) if log_axis else v))
dims.append(dict(label="CV mean MSE", values=vals))
fig = go.Figure(go.Parcoords(
    line=dict(color=vals, colorscale=[[0, SPLIT_COLORS["validation"]], [1, SPLIT_COLORS["neutral"]]],
              showscale=True, colorbar=dict(title="CV MSE")),
    dimensions=dims))
fig.update_layout(title="Hyperparameters vs. objective (darker = lower CV MSE)", width=1100, height=520)
fig.show()
fig.write_html(str(HTML_DIR / "C3_08_parallel_coordinates.html"), include_plotlyjs="inline")

## 9. Parameter importance

In [ ]:
try:
    importances = optuna.importance.get_param_importances(study)
except Exception as e:           # needs enough completed trials
    importances = {}
    print(f"parameter importance not available: {e}")
if importances:
    names = list(importances)[::-1]
    fig = go.Figure(go.Bar(x=[importances[k] for k in names], y=names, orientation="h",
                           marker_color=SPLIT_COLORS["neutral"],
                           marker_line=dict(color=SPLIT_COLORS["reference"], width=1)))
    fig.update_layout(title="Hyperparameter importance (fANOVA, share of objective variance)",
                      xaxis_title="importance", width=800, height=450)
    fig.show()
    fig.write_html(str(HTML_DIR / "C3_09_param_importance.html"), include_plotlyjs="inline")

## 10. Best configuration — a closer look

Per-fold and per-output spread for the winning trial (not just its
mean), its physics penalties, and a reminder to check HC-λ convexity by
hand if `activation == "relu"` won.

In [ ]:
best_fold_mses = best.user_attrs["fold_mses"]
best_rows = [{"item": "trial", "value": str(best.number)},
             {"item": "CV mean MSE", "value": f"{study.best_value:.6f}"},
             {"item": "CV std MSE", "value": f"{best.user_attrs['std_mse']:.6f}"},
             {"item": "hidden_units", "value": str(best.user_attrs["hidden_units"])},
             {"item": "activation", "value": best.user_attrs["activation"]}]
best_rows += [{"item": k, "value": f"{v:.6g}" if isinstance(v, float) else str(v)} for k, v in best.params.items()]
best_table = pl.DataFrame(best_rows)
simple_table_html(best_table, "C3 -- Best configuration", HTML_DIR / "C3_10_best_config.html")

fig = make_subplots(rows=1, cols=3, subplot_titles=["validation MSE per fold", "validation MSE per output",
                                                    "physics penalty per constraint (unscaled)"])
fig.add_trace(go.Bar(x=[f"fold {i}" for i in range(len(best_fold_mses))], y=best_fold_mses,
                     marker_color=SPLIT_COLORS["validation"], showlegend=False), row=1, col=1)
mpo = best.user_attrs["mse_per_output"]
fig.add_trace(go.Bar(x=list(mpo), y=list(mpo.values()), marker_color=[VARIABLE_COLORS[o] for o in mpo],
                     showlegend=False), row=1, col=2)
pen = best.user_attrs["physics_penalty"]
fig.add_trace(go.Bar(x=list(pen), y=[max(v, 1e-12) for v in pen.values()],
                     marker_color=[CONSTRAINT_STYLE[n]["color"] for n in pen], showlegend=False), row=1, col=3)
fig.update_yaxes(type="log", row=1, col=3)
fig.update_layout(title=f"C3 best trial #{best.number}: spread and physics compliance", width=1150, height=420)
fig.show()
fig.write_html(str(HTML_DIR / "C3_10_best_detail.html"), include_plotlyjs="inline")

if best.user_attrs["activation"] == "relu":
    print("NOTE: best trial uses ReLU -- HC-lambda convexity (Eq. 3.11) will show near-zero penalty")
    print("regardless of whether the learned HC(lambda) shape is actually convex. Check it before trusting it.")
best_table

## Persist outputs

`C3_best_config.json` is what C4 reads. Besides the tuned values it
records the architecture actually used (`hidden_units`, `activation`),
whether it came from B1 or from this search, and the schedule settings
the search ran with.

In [ ]:
best_config = dict(best.params)
best_config.update({
    "hidden_units": best.user_attrs["hidden_units"],
    "activation": best.user_attrs["activation"],
    "architecture_source": "C3 search" if SEARCH_ARCHITECTURE else f"B1 ({selection['architecture_id']})",
    "cv_mean_mse": study.best_value,
    "cv_fold_mses": best_fold_mses,
    "physics_penalty": best.user_attrs["physics_penalty"],
    "n_trials": N_TRIALS, "epochs_per_trial": EPOCHS_PER_TRIAL, "patience_per_trial": PATIENCE_PER_TRIAL,
    "ramp_full_fraction": RAMP_FULL_FRACTION,
    "c2_calibration_mode": loss_config["calibration_mode"],
})
with open(OUT_DIR / "C3_best_config.json", "w") as f:
    json.dump(best_config, f, indent=2)
print(f"Saved to {OUT_DIR}")

## Next

**C4** trains the final 10-model ensemble using this configuration —
B1's architecture (`hidden_units`, `tanh`), learning rate, batch size,
dropout, weight decay and the scaled physics weights — with the full
epoch/patience budget (Adam, then L-BFGS) that C3's search deliberately
shortened, and C2's schedule at its original time scale. After C4, the
architecture sensitivity check (PINN trained on one or two architectures
B1 found statistically equivalent) confirms that the conclusion about
the physics does not hinge on the choice of backbone.